# Step 4: RAG Pipeline — Retrieval + Groq LLM

This is where everything comes together. We:
1. Load the ChromaDB vector database from Step 3
2. Connect to Groq API (free tier, no credit card)
3. Build the RAG pipeline: **User Question → Retrieve Chunks → Build Prompt → LLM → Answer**
4. Test with real financial questions

**LLM:** Llama 3.3 70B via Groq (free tier, ~500 tokens/sec)  
**Get your API key:** https://console.groq.com

## 4.1 Install Dependencies

In [1]:
!pip install groq chromadb

## 4.2 Configuration & API Key Setup

**How to get a free API key:**
1. Go to https://console.groq.com
2. Sign up with email or Google account (no credit card needed)
3. Go to API Keys section
4. Click "Create API Key"
5. Copy the key and paste it below

In [ ]:
import os
import chromadb
from groq import Groq

# ========== PASTE YOUR GROQ API KEY HERE ==========
GROQ_API_KEY = "your-groq-api-key-here"  # <-- Replace this
# ===================================================

# Initialize Groq client
client = Groq(api_key=GROQ_API_KEY)

# Model to use (free tier)
# llama-3.3-70b-versatile — best quality for RAG on free tier
MODEL_NAME = "llama-3.3-70b-versatile"

# ChromaDB path — must match Step 3
CHROMA_DB_DIR = os.path.join("data", "chroma_db")
COLLECTION_NAME = "sec_filings"

print(f"Model: {MODEL_NAME}")
print(f"ChromaDB: {os.path.abspath(CHROMA_DB_DIR)}")

Model: llama-3.3-70b-versatile
ChromaDB: c:\Users\anfas\Downloads\finance report rag\data\chroma_db


## 4.3 Load ChromaDB Collection

In [3]:
# Connect to the existing ChromaDB database from Step 3
chroma_client = chromadb.PersistentClient(path=CHROMA_DB_DIR)
collection = chroma_client.get_collection(name=COLLECTION_NAME)

print(f"Loaded collection '{COLLECTION_NAME}'")
print(f"Total chunks: {collection.count()}")

Loaded collection 'sec_filings'
Total chunks: 5786


## 4.4 Test Groq Connection

Quick check to make sure the API key works.

In [4]:
# Quick test — if this fails, check your API key
test_response = client.chat.completions.create(
    messages=[{"role": "user", "content": "Say 'Groq is connected' in exactly 3 words."}],
    model=MODEL_NAME,
    max_tokens=20,
)

print(f"Groq says: {test_response.choices[0].message.content}")
print(f"\n✓ Groq API connection successful!")
print(f"Model: {test_response.model}")
print(f"Tokens used: {test_response.usage.total_tokens}")

Groq says: Groq is connected

✓ Groq API connection successful!
Model: llama-3.3-70b-versatile
Tokens used: 53


## 4.5 Build the RAG Pipeline

The pipeline:
1. Takes a user question
2. Retrieves the most relevant chunks from ChromaDB
3. Builds a prompt with the retrieved context
4. Sends it to Groq (Llama 3.3 70B)
5. Returns a grounded answer with source citations

In [5]:
# System prompt — tells the LLM how to behave as a financial analyst
SYSTEM_PROMPT = """You are an expert financial analyst assistant. Your job is to answer 
questions about company SEC filings (10-K and 20-F annual reports) based ONLY on the 
provided context.

Rules:
1. Answer ONLY based on the provided context. Do not use outside knowledge.
2. If the context does not contain enough information to answer, say so clearly.
3. Always cite which company's filing the information comes from.
4. When discussing financial figures, be precise — include exact numbers from the context.
5. If the question asks to compare companies, use data from all relevant companies in the context.
6. Structure your response clearly with key findings first, then supporting details.
"""


def retrieve_context(question, n_results=5, ticker_filter=None):
    """
    Retrieve relevant chunks from ChromaDB.
    
    Args:
        question: user's natural language query
        n_results: number of chunks to retrieve
        ticker_filter: optional company ticker to filter by
    
    Returns:
        formatted context string and metadata list
    """
    where_filter = None
    if ticker_filter:
        where_filter = {"ticker": ticker_filter}
    
    results = collection.query(
        query_texts=[question],
        n_results=n_results,
        where=where_filter,
    )
    
    # Format retrieved chunks as context
    context_parts = []
    sources = []
    
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0],
    )):
        source_label = f"{meta['ticker']} ({meta['company_name']}) - {meta['form_type']}"
        context_parts.append(
            f"[Source {i+1}: {source_label}]\n{doc}"
        )
        sources.append({
            'source': source_label,
            'distance': dist,
            'chunk_index': meta['chunk_index'],
        })
    
    context = "\n\n---\n\n".join(context_parts)
    return context, sources


def ask_question(question, n_results=5, ticker_filter=None, show_sources=True):
    """
    Full RAG pipeline: retrieve context → build prompt → call Groq → return answer.
    
    Args:
        question: user's question about the SEC filings
        n_results: number of chunks to retrieve (more = broader context)
        ticker_filter: optional company ticker to scope the search
        show_sources: whether to print source metadata
    
    Returns:
        answer string
    """
    # Step 1: Retrieve relevant chunks
    context, sources = retrieve_context(question, n_results, ticker_filter)
    
    # Step 2: Build the prompt
    user_prompt = f"""Context from SEC filings:

{context}

---

Question: {question}

Provide a detailed answer based on the context above."""
    
    # Step 3: Call Groq
    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_prompt},
        ],
        model=MODEL_NAME,
        temperature=0.2,  # low temperature for factual, consistent answers
        max_tokens=1024,
    )
    
    answer = response.choices[0].message.content
    
    # Step 4: Display results
    print(f"Question: {question}")
    if ticker_filter:
        print(f"Company filter: {ticker_filter}")
    print(f"Chunks retrieved: {n_results}")
    print("=" * 60)
    print(f"\n{answer}")
    
    if show_sources:
        print("\n" + "=" * 60)
        print("Sources used:")
        for s in sources:
            print(f"  - {s['source']} (chunk #{s['chunk_index']}, distance: {s['distance']:.4f})")
    
    # Show token usage
    print(f"\nTokens used: {response.usage.total_tokens} "
          f"(prompt: {response.usage.prompt_tokens}, "
          f"completion: {response.usage.completion_tokens})")
    
    return answer

print("RAG pipeline ready.")

RAG pipeline ready.


## 4.6 Test the RAG Pipeline

Let's test with real financial questions across different types:
- Company-specific questions
- Cross-company comparisons
- Risk factor analysis
- Financial metrics

In [6]:
# Test 1: Company-specific revenue question
_ = ask_question(
    "What is Microsoft's total revenue and how has it changed year over year?",
    ticker_filter="MSFT",
)

Question: What is Microsoft's total revenue and how has it changed year over year?
Company filter: MSFT
Chunks retrieved: 5

**Key Findings:**
Microsoft's total revenue for fiscal year 2025 is $281,724 million, representing a 15% increase from the previous year. (Source 2: MSFT - 10-K)

**Supporting Details:**
According to the SUMMARY RESULTS OF OPERATIONS table in Source 2, Microsoft's revenue for fiscal year 2025 is $281,724 million, compared to $245,122 million in fiscal year 2024. This represents a year-over-year increase of $36,602 million, or 15%. (Source 2: MSFT - 10-K)

Additionally, another part of the context mentions that revenue increased $14.0 billion or 13% (Source 1: MSFT - 10-K), but this appears to be a different metric or time frame, as the increase mentioned in Source 2 is $36.6 billion or 15%. 

It is also worth noting that the revenue growth is driven by growth across each of Microsoft's segments, including Intelligent Cloud, Productivity and Business Processes, an

In [7]:
# Test 2: Risk factors across companies
_ = ask_question(
    "What are the key risk factors related to cybersecurity mentioned in the filings?",
    n_results=7,
)

Question: What are the key risk factors related to cybersecurity mentioned in the filings?
Chunks retrieved: 7

**Key Findings:**
The key risk factors related to cybersecurity mentioned in the filings include risks from intentional acts of hackers, state-sponsored organizations, and competitors, as well as unintentional acts or omissions by customers, contractors, business partners, vendors, employees, and other third parties. Additionally, errors in processes or technologies and the increased use of AI and remote work arrangements are also cited as potential cybersecurity risks.

**Supporting Details:**

1. **IBM (IBM) - 10-K**: The company faces numerous and evolving cybersecurity threats, including risks originating from the increased use of AI, intentional acts of individual and groups of criminal hackers, hacktivists, state-sponsored organizations, nation states, and competitors. (Source 1 and Source 3)
2. **CTSH (Cognizant) - 10-K**: Although Cognizant did not identify any materi

In [8]:
# Test 3: Company comparison
_ = ask_question(
    "Compare the cloud computing strategies of Microsoft and Google based on their filings.",
    n_results=8,
)

Question: Compare the cloud computing strategies of Microsoft and Google based on their filings.
Chunks retrieved: 8

**Key Findings:**

1. Both Microsoft and Google have a strong focus on cloud computing, with each company highlighting the importance of their cloud services in their respective filings.
2. Microsoft's cloud strategy is centered around its Azure platform, as well as its Microsoft 365 Commercial cloud services, which drove revenue growth in fiscal year 2025 (Source 2: MSFT - 10-K, Source 6: MSFT - 10-K).
3. Google's cloud strategy is focused on its Google Cloud Platform, Google Workspace, and other enterprise services, with a emphasis on developing and deploying enterprise-ready cloud services (Source 4: GOOGL - 10-K, Source 7: GOOGL - 10-K).
4. Both companies are investing heavily in their cloud infrastructure, with Microsoft highlighting the importance of its datacenter operations and Google noting the significant costs and liabilities associated with building and main

In [9]:
# Test 4: Indian IT company specific
_ = ask_question(
    "What are the main business segments and their revenue contribution?",
    ticker_filter="INFY",
)

Question: What are the main business segments and their revenue contribution?
Company filter: INFY
Chunks retrieved: 5

**Key Findings:**
The main business segments of Infosys (INFY) are Financial Services, Manufacturing, Energy, Utilities, Resources and Services, Retail, Communication, Hi-Tech, Life Sciences, and All other segments. The revenue contribution of these segments for fiscal 2026 and fiscal 2025 is provided in the context.

**Supporting Details:**
According to the context from INFY's 20-F filing (Source 2), the revenue contribution of each business segment for fiscal 2026 and fiscal 2025 is as follows:

1. **Financial Services**: 27.9% (fiscal 2026) and 27.7% (fiscal 2025)
2. **Manufacturing**: 16.3% (fiscal 2026) and 15.5% (fiscal 2025)
3. **Energy, Utilities, Resources and Services**: 13.3% (both fiscal 2026 and fiscal 2025)
4. **Retail**: 12.9% (fiscal 2026) and 13.5% (fiscal 2025)
5. **Communication**: 12.2% (fiscal 2026) and 11.7% (fiscal 2025)
6. **Hi-Tech**: 7.8% (fi

In [10]:
# Test 5: Employee and workforce related
_ = ask_question(
    "How many employees do these IT companies have and what are their workforce strategies?",
    n_results=7,
)

Question: How many employees do these IT companies have and what are their workforce strategies?
Chunks retrieved: 7

**Key Findings:**
The provided context does not contain information on the exact number of employees for the mentioned IT companies (Accenture, Infosys, and Wipro). However, it discusses workforce strategies and utilization rates for Infosys and Wipro.

**Supporting Details:**

1. **Employee Utilization Rates (Infosys):** According to Infosys' 20-F filing (Source 2), the company defines employee utilization for IT services as the proportion of total billed person months to total available person months, excluding sales, administrative, and support personnel. The filing mentions that an unanticipated termination of a significant project could cause lower utilization, and that employees are not utilized when they are enrolled in training programs.

2. **Workforce Strategies (Infosys):** Infosys manages utilization by monitoring project requirements and timetables. The num

In [11]:
_ = ask_question(
    "Your custom question here",
    ticker_filter="MSFT",  # or remove this line for all companies
)

Question: Your custom question here
Company filter: MSFT
Chunks retrieved: 5

Based on the provided context, there is no specific question to answer. However, I can provide an analysis of the context.

The context appears to be excerpts from Microsoft's (MSFT) 10-K filings, specifically from PART I, Item 1A, and PART II, Item 7. The page numbers mentioned are 28, 26, 25, 36, and 23.

Key findings:
- All excerpts are from Microsoft's (MSFT) 10-K filings.
- The excerpts are from different page numbers, indicating they may be from different sections or years of the filings.
- The majority of the excerpts (4 out of 5) are from PART I, Item 1A, which typically discusses business risks and risk factors.
- One excerpt is from PART II, Item 7, which typically discusses management's discussion and analysis (MD&A) of financial condition and results of operations.

Supporting details:
- Since the context only provides page numbers and section references, it is not possible to provide more detaile

In [13]:
_ = ask_question(
    "can u calcultate quick ratio",
    ticker_filter="MSFT",  # or remove this line for all companies
)

Question: can u calcultate quick ratio
Company filter: MSFT
Chunks retrieved: 5

**Key Findings:**
To calculate the quick ratio, we need to know the company's current assets, current liabilities, and inventory. However, the provided context from MSFT (Microsoft) - 10-K does not contain enough information to directly calculate the quick ratio.

**Supporting Details:**
The quick ratio, also known as the acid-test ratio, is calculated as follows:

Quick Ratio = (Current Assets - Inventory) / Current Liabilities

From the provided context, we can see that there are some current assets and liabilities mentioned, but not all the necessary information is available. For example:

- **Current Assets:** Some components of current assets are mentioned, such as "Short-term investments" and "Other current assets" in Source 5, but the total current assets are not provided.
- **Inventory:** There is no information about the company's inventory in the provided context.
- **Current Liabilities:** Some 

## 4.7 Interactive Q&A Loop

Run this cell to ask your own questions interactively.

In [14]:
print("Financial Report Analyzer — Interactive Mode")
print("Available companies: MSFT, GOOGL, ACN, IBM, CTSH, INFY, WIT")
print("Type 'quit' to exit\n")

while True:
    question = input("\nYour question: ").strip()
    
    if question.lower() in ['quit', 'exit', 'q']:
        print("Goodbye!")
        break
    
    if not question:
        continue
    
    # Ask if they want to filter by company
    company = input("Filter by company ticker (or press Enter for all): ").strip().upper()
    ticker_filter = company if company else None
    
    print()
    _ = ask_question(question, ticker_filter=ticker_filter)
    print()

Financial Report Analyzer — Interactive Mode
Available companies: MSFT, GOOGL, ACN, IBM, CTSH, INFY, WIT
Type 'quit' to exit


Question: who are you
Company filter: HI WHERE CAN ISEE THE ANSWER
Chunks retrieved: 5

Based on the provided context, there is no information available to answer the question "who are you". The context appears to be empty, with no details or data from any company's SEC filings (10-K or 20-F annual reports). 

Therefore, I must state that the context does not contain enough information to answer the question.

Sources used:

Tokens used: 262 (prompt: 194, completion: 68)


Question: which all company data is available
Chunks retrieved: 5

Based on the provided context, the following companies' data is available:

1. **INFY (Infosys)**: The context provides information about Infosys from three different sources (Source 1, Source 3, and Source 5). The data includes:
	* A list of subsidiaries and their respective countries, with ownership percentages (Source 1 and

KeyboardInterrupt: Interrupted by user

---

## ✅ Step 4 Complete

**What we built:**
- A complete RAG pipeline: Question → ChromaDB Retrieval → Prompt Construction → Groq (Llama 3.3 70B) → Grounded Answer
- Company-specific filtering (ask about one company or compare across all)
- Source citation showing which filing and chunk each answer came from
- Token usage tracking per query
- Interactive Q&A mode

**Why Groq for interviews:**
- "I used Groq for inference because it runs open-source models (Llama 3.3 70B) at 500+ tokens/sec on custom LPU hardware"
- "The system is LLM-agnostic — switching from Groq to OpenAI or Gemini only requires changing the API client, not the pipeline"
- "Groq's API is OpenAI-compatible, so the same code works with any OpenAI-compatible provider"

**What to highlight in interviews:**
- "The LLM only sees retrieved context, not the full document — this controls hallucination"
- "I use a system prompt that enforces citation and prevents the model from using outside knowledge"
- "I set temperature=0.2 for factual consistency in financial analysis"

**Next step:** `Step5_Agent_and_Tools.ipynb` — add tools (financial ratio calculator, trend analysis) and wrap everything in a LangGraph agent